In [14]:
import os
from dotenv import load_dotenv

#langchain
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [35]:
load_dotenv()

True

In [36]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")



## Loading our data


In [37]:
DATA_FILE_PATH = os.path.join('data','hr_policy.txt')

## Data Ingestion

In [6]:
loader =TextLoader(DATA_FILE_PATH, encoding='utf-8')

documents= loader.load()
print("Data loaded")

print("=" * 40)
print(documents)

Data loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content="Here is a structured, clean HR Policy Knowledge Base formatted specifically for building a RAG (Retrieval-Augmented Generation) system.\nTo make chunking and retrieval accurate for your AI, each policy section below uses clear headers, self-contained paragraphs, and explicit rules so the LLM can answer employee questions without hallucinating.\n[COMPANY NAME] — COMPLETE HR POLICY HANDBOOK\nVersion: 2.0 | Effective Date: January 1, 2026 | Jurisdiction: India\n\n==================================================\nSECTION 1: PROBATION & CONFIRMATION POLICY\n\n1.1 Probation Period Duration\nAll new full-time employees undergo a mandatory probation period of 6 months from their date of joining. Management reserves the right to extend the probation period by up to 3 months if performance or conduct does not meet company standards.\n1.2 Performance Evaluation During Probation\nEmployees on probation are evaluated 

#### Langchain documents

Langchain processes everything in from of documents

Documents:
 * page conetent-- the actual data
 * metadata --extra information about the data

In [7]:
len(documents)

1

In [16]:
print(documents[0].page_content)

Here is a structured, clean HR Policy Knowledge Base formatted specifically for building a RAG (Retrieval-Augmented Generation) system.
To make chunking and retrieval accurate for your AI, each policy section below uses clear headers, self-contained paragraphs, and explicit rules so the LLM can answer employee questions without hallucinating.
[COMPANY NAME] — COMPLETE HR POLICY HANDBOOK
Version: 2.0 | Effective Date: January 1, 2026 | Jurisdiction: India

SECTION 1: PROBATION & CONFIRMATION POLICY

1.1 Probation Period Duration
All new full-time employees undergo a mandatory probation period of 6 months from their date of joining. Management reserves the right to extend the probation period by up to 3 months if performance or conduct does not meet company standards.
1.2 Performance Evaluation During Probation
Employees on probation are evaluated at the 3-month and 6-month marks by their reporting manager. Upon successful completion of the 6-month review, HR will issue a formal Confirma

In [17]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


## splitting our data

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap =50 
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='Here is a structured, clean HR Policy Knowledge Base formatted specifically for building a RAG (Retrieval-Augmented Generation) system.\nTo make chunking and retrieval accurate for your AI, each policy section below uses clear headers, self-contained paragraphs, and explicit rules so the LLM can answer employee questions without hallucinating.\n[COMPANY NAME] — COMPLETE HR POLICY HANDBOOK\nVersion: 2.0 | Effective Date: January 1, 2026 | Jurisdiction: India'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='==================================================\nSECTION 1: PROBATION & CONFIRMATION POLICY'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1.1 Probation Period Duration\nAll new full-time employees undergo a mandatory probation period of 6 months from their date of joining. Management reserves the right to extend the probation period by up to 3 months if performance or co

In [10]:
len(chunks)

30

## Now each spilted chunk is a document - page content and metadata

In [11]:
print(chunks[0])

page_content='Here is a structured, clean HR Policy Knowledge Base formatted specifically for building a RAG (Retrieval-Augmented Generation) system.
To make chunking and retrieval accurate for your AI, each policy section below uses clear headers, self-contained paragraphs, and explicit rules so the LLM can answer employee questions without hallucinating.
[COMPANY NAME] — COMPLETE HR POLICY HANDBOOK
Version: 2.0 | Effective Date: January 1, 2026 | Jurisdiction: India' metadata={'source': 'data\\hr_policy.txt'}


In [12]:
print(chunks[5])

page_content='==================================================
SECTION 2: LEAVE POLICY' metadata={'source': 'data\\hr_policy.txt'}


In [13]:
print(chunks[8].page_content)

2.3 Leave Carry Forward and Encashment
• Casual Leave (CL) and Sick Leave (SL) lapse at the end of the calendar year (December 31) and cannot be carried forward.
• Up to 15 unused Earned Leaves (EL) can be carried forward to the next calendar year, up to a maximum accumulation cap of 45 days. Leave encashment is applicable only at the time of full and final settlement upon resignation.
2.4 Maternity and Paternity Leave


In [15]:
from langchain_community.embeddings import JinaEmbeddings


embeddings_model =JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB Model ready the name is ",embeddings_model.model_name)

EMB Model ready the name is  jina-embeddings-v2-base-en


### Store data in vector DB

In [16]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, embeddings_model)

print("chunks are stored ",vector_store.index.ntotal)

chunks are stored  30


In [17]:
test_query = "How many sick leaves employess get ?"

## similarty search 
top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query : {test_query}\n")

for i,match in enumerate(top_matches,start=1):
    print(f"---Match {i}---")
    print(match.page_content)
    print()


Query : How many sick leaves employess get ?

---Match 1---
2.3 Leave Carry Forward and Encashment
• Casual Leave (CL) and Sick Leave (SL) lapse at the end of the calendar year (December 31) and cannot be carried forward.
• Up to 15 unused Earned Leaves (EL) can be carried forward to the next calendar year, up to a maximum accumulation cap of 45 days. Leave encashment is applicable only at the time of full and final settlement upon resignation.
2.4 Maternity and Paternity Leave

---Match 2---
2.1 Annual Leave Entitlement
Confirmed full-time employees are entitled to the following leaves per calendar year (January to December):
• Earned Leave (EL) / Privilege Leave (PL): 18 days per year (credited at 1.5 days per month).
• Casual Leave (CL): 6 days per year (credited at 0.5 days per month).
• Sick Leave (SL): 6 days per year (credited at 0.5 days per month).
• Public Holidays: 10 fixed national and festival holidays as published in the annual holiday calendar.



## Data Retrival

* LLM


In [34]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0   # creativity
)

llm.model_name

'openai/gpt-oss-120b'

In [20]:
test_response = llm.invoke("Hey is learning Rag hard ? answer in 1 line")

In [21]:
test_response.content

'Learning RAG can be challenging at first, but with clear resources and hands‑on practice it becomes manageable.'

### AI Agent

* LLM -- Brain
* Tool -- Super power
* memory -- no memory

In [22]:
from langchain.agents import create_agent

## Tool

In [43]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str)-> str:
    """
    Search the HR policy document for information about leave,work from home,notice period etc 
    """
    matching_chunks= retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks )

In [ ]:
hr_assistant = create_agent(
    model= llm,
    tools=[search_hr_policy],
        system_prompt="""
        You are a friendly HR assistent.
        Always use the search_hr_policy tool to look up
        facts before answering.
        IF the answer isn't in the search results, say you don't know "
        instead of guessing."
        """
)

print("HR assistent agent is ready to answer question!")

HR assistent agent is ready to answer question!


In [45]:
def ask_hr_assistant(question:str) -> str:
    """Send a questions to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("Question:", question)
    print("=" * 60)

    response = hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer = response['messages'][-1].content
    print("Answer:",answer)
    print("="*60)
    print()
    return answer

In [50]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"Tell me about leave policy?"
            }
        ]
    }
)

In [51]:
response

{'messages': [HumanMessage(content='Tell me about leave policy?', additional_kwargs={}, response_metadata={}, id='427e31a4-c1eb-44c7-816e-b8e3bb9b763b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer about leave policy. Must use search_hr_policy tool to look up facts before answering. So we need to query the HR policy. Use the tool with question "leave policy".', 'tool_calls': [{'id': 'fc_28aed42a-7be2-4eea-8af6-d0db8f2f4b13', 'function': {'arguments': '{"question":"leave policy"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 188, 'total_tokens': 257, 'completion_time': 0.145217247, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.007496798, 'prompt_tokens_details': None, 'queue_time': 0.318438542, 'total_time': 0.152714045}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_64b2f1c926', 'service_tier': 'on_demand', 'finish_reason

In [52]:
response['messages'][-1].content

'**Leave Policy (as per the HR policy document)**  \n\n**1. Annual Leave Entitlement (January\u202f–\u202fDecember)**  \n| Leave Type | Total Days per Year | Accrual Rate |\n|------------|--------------------|--------------|\n| Earned Leave (EL) / Privilege Leave (PL) | **18 days** | 1.5 days\u202fper\u202fmonth |\n| Casual Leave (CL) | **6 days** | 0.5 days\u202fper\u202fmonth |\n| Sick Leave (SL) | **6 days** | 0.5 days\u202fper\u202fmonth |\n| Public Holidays | **10 days** (fixed national/festival holidays) | – |\n\n**2. Carry‑Forward & Encashment**  \n- **Casual Leave (CL)** and **Sick Leave (SL)** **expire** on\u202f31\u202fDecember each year; they **cannot be carried forward**.  \n- **Earned Leave (EL)**: Up to **15 unused days** may be carried forward to the next calendar year, subject to a **maximum accumulation cap of 45 days**.  \n- **Leave encashment** is only permitted **at the time of full and final settlement** when an employee resigns.\n\n**3. Maternity & Paternity Leave

In [53]:
response['messages'][-2].content

'==================================================\nSECTION 2: LEAVE POLICY\n\n2.3 Leave Carry Forward and Encashment\n• Casual Leave (CL) and Sick Leave (SL) lapse at the end of the calendar year (December 31) and cannot be carried forward.\n• Up to 15 unused Earned Leaves (EL) can be carried forward to the next calendar year, up to a maximum accumulation cap of 45 days. Leave encashment is applicable only at the time of full and final settlement upon resignation.\n2.4 Maternity and Paternity Leave\n\n2.1 Annual Leave Entitlement\nConfirmed full-time employees are entitled to the following leaves per calendar year (January to December):\n• Earned Leave (EL) / Privilege Leave (PL): 18 days per year (credited at 1.5 days per month).\n• Casual Leave (CL): 6 days per year (credited at 0.5 days per month).\n• Sick Leave (SL): 6 days per year (credited at 0.5 days per month).\n• Public Holidays: 10 fixed national and festival holidays as published in the annual holiday calendar.'